# Step 9 (Phase 4) — Escalation Predictor

Trains an XGBoost model to answer: "given an incident's first few readings,
will it turn out severe?" Uses the 5-clue table built by
`src/ml/escalation_prep.py` (Phase 4, Step 2).

Trains on 824 early-window readings from 85 incidents.
Tests on 259 sealed readings from 22 different, later incidents —
never seen during training.

## Cell 1 — Load the 4 files

Loads the practice table (`X_train`/`y_train`) and the sealed exam table
(`X_test`/`y_test`) saved earlier, plus the label sheet that tells us what
each of the 5 columns actually means.

In [13]:
import json
from pathlib import Path

import numpy as np
import xgboost as xgb
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

import os
os.chdir(Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd())

ML_DIR = Path("ml_models")

X_train = np.load(ML_DIR / "X_escalation_train.npy")
X_test  = np.load(ML_DIR / "X_escalation_test.npy")
y_train = np.load(ML_DIR / "y_escalation_train.npy")
y_test  = np.load(ML_DIR / "y_escalation_test.npy")

with open(ML_DIR / "escalation_feature_cols.json") as f:
    feature_cols = json.load(f)

print("Train:", X_train.shape, "  Test:", X_test.shape)
print("Feature order:", feature_cols)

Train: (1354, 7)   Test: (424, 7)
Feature order: ['votes', 'ensemble_score', 'zscore_value', 'iforest_score', 'lstm_error', 'ensemble_score_trend', 'consecutive_anomalous_ticks']


## Cell 2 — Sanity check before training

Just double-checks the loaded files match what `escalation_prep.py` printed
earlier (824 train rows / 200 severe, 259 test rows / 96 severe). A quick
habit to catch a stale or wrong file before wasting time training on it.

In [14]:
print(f"Train: {y_train.sum()} severe / {len(y_train)} rows ({100*y_train.mean():.1f}%)")
print(f"Test:  {y_test.sum()} severe / {len(y_test)} rows ({100*y_test.mean():.1f}%)")

Train: 324 severe / 1354 rows (23.9%)
Test:  156 severe / 424 rows (36.8%)


## Cell 3 — Train the model

`scale_pos_weight` tells the model "treat getting a severe case wrong as
roughly N times more costly than getting a non-severe case wrong" — this
stops it from taking the lazy shortcut of always guessing "not severe"
(which would already be right ~76% of the time, but useless).

`max_depth=3` keeps each tree shallow on purpose. With only 824 rows and
5 clues, a deep tree would just memorize the practice set instead of
learning a real pattern — like a student who memorizes the exact practice
questions instead of understanding the topic, then fails on new questions.

In [15]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=100,       # how many trees to build
    max_depth=3,            # keep trees shallow — avoid memorizing 824 rows
    learning_rate=0.1,      # how big a correction each tree is allowed to make
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
)

model.fit(X_train, y_train)
print("Trained.")

scale_pos_weight = 3.18
Trained.


## Cell 4 — Grade it honestly against the sealed test rows

This is the only place the model's guesses ever get compared to the 259
test answers. Precision, recall, and F1 mean the same thing here as they
did in Phase 3's `05_evaluation.ipynb` — just applied to "will this incident
turn severe" instead of "is this reading anomalous."

In [16]:
y_pred = model.predict(X_test)

print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print("\nConfusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))

Precision: 0.384
Recall:    0.487
F1:        0.429

Confusion matrix (rows=actual, cols=predicted):
[[146 122]
 [ 80  76]]


## Cell 5 — Save the trained model

Same pattern as your Z-Score/Isolation Forest/LSTM files in `ml_models/` —
a model file plus a small config file recording which 5 columns it expects
and in what order, so nothing gets mixed up when it's loaded again later.


In [17]:
model.save_model(ML_DIR / "xgb_escalation.json")

with open(ML_DIR / "xgb_escalation_config.json", "w") as f:
    json.dump({"feature_cols": feature_cols, "scale_pos_weight": scale_pos_weight}, f, indent=2)

print("Saved xgb_escalation.json + xgb_escalation_config.json to ml_models/")

Saved xgb_escalation.json + xgb_escalation_config.json to ml_models/


## Cell 6 — Which clues did the model actually lean on?

`model.feature_importances_` gives one number per clue, showing roughly how
often that clue was used to make a split across all the trees. Higher =
the model leaned on it more. If everything comes out roughly equal and low,
that's a sign the model didn't find much of a real pattern in any single
clue — more evidence it's closer to guessing than learning.

In [18]:
importances = model.feature_importances_

ranked = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

print("Feature importance (higher = leaned on more):")
for name, score in ranked:
    bar = "█" * int(score * 50)
    print(f"  {name:<16} {score:.3f}  {bar}")

Feature importance (higher = leaned on more):
  votes            0.216  ██████████
  lstm_error       0.200  █████████
  iforest_score    0.169  ████████
  ensemble_score_trend 0.162  ████████
  zscore_value     0.133  ██████
  consecutive_anomalous_ticks 0.120  ██████
  ensemble_score   0.000  
